# Investment Agent — Diagnostics Notebook

Test each pipeline component **individually** or run the full pipeline. Each section is self-contained with clear **PASS** / **FAIL** output.

## How to run

| Mode | Steps |
|------|-------|
| **Full run** | Run all cells top-to-bottom (Kernel → Run All) |
| **Single component** | Run **Section 0** (Environment) + **Section 0.5** (Quick Setup), then run the section you want to test |
| **From section N onward** | Run Section 0, Section 0.5, then Section 1 through N |

**Pipeline:** Config → Data → Indicators → Fundamentals → **Macro Intelligence** → Finnhub → Alpha Vantage → Strategies → Moderation → Risk → Execution → Journal → Performance & Trade Outcomes → Notifications → Orchestrator → Backtesting → Walk-Forward Validation

**Data Rationale:** [docs/DATA_RATIONALE.md](../docs/DATA_RATIONALE.md) | **Backtesting:** [docs/BACKTESTING.md](../docs/BACKTESTING.md) | **Walk-Forward:** [docs/BACKTESTING.md#walk-forward-validation](../docs/BACKTESTING.md#walk-forward-validation)

---
## 0. Environment Setup

In [1]:
import sys, os, json
from pathlib import Path
from datetime import datetime

# Ensure project root is on sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version}")
print(f"Timestamp    : {datetime.utcnow().isoformat()}Z")

Project root : /Users/Kayvan/Library/Mobile Documents/com~apple~CloudDocs/AI_Projects/Investment-agent
Python       : 3.11.13 (main, Jun  5 2025, 08:14:07) [Clang 14.0.6 ]
Timestamp    : 2026-03-06T16:36:29.391334Z


---
## 0.5 Quick Setup (Run before testing individual sections)

Creates shared objects so you can run any section below without executing all prior cells:

- **Data:** `fetcher`, `df`, `TEST_TICKER`, `analysis`, `macro`, `stocks_data`, `sub_results`
- **Test fixtures:** `test_trades`, `sample_portfolio`
- **Placeholders:** `panel_results`, `risk_verdicts`, `exec_results` (populated when Moderation, Risk, Order Manager run)

In [2]:
# Quick Setup: create shared objects for standalone section testing
import json
from src.agents.market_data.data_fetcher import DataFetcher

fetcher = DataFetcher()
TEST_TICKER = "AAPL"
MULTI_TICKERS = ["AAPL", "MSFT", "GOOGL"]

# Minimal data needed by later sections
df = fetcher.get_ohlcv(TEST_TICKER, period="1y")
macro = fetcher.get_macro_data()
analysis = fetcher.get_stock_analysis(TEST_TICKER) if not df.empty else {}

# Multi-ticker data (for strategy, moderation, risk, execution sections)
stocks_data = []
for t in MULTI_TICKERS:
    try:
        d = fetcher.get_stock_analysis(t)
        d["ticker"] = t
        stocks_data.append(d)
    except Exception:
        stocks_data.append({"ticker": t, "indicators": {}, "fundamentals": {}})

from src.agents.strategy.engine import StrategyEngine
engine = StrategyEngine()
sub_results = engine.run_sub_strategies(stocks_data, existing_positions=set())

# Default test trades (used by moderation, risk, execution, journal sections)
test_trades = [
    {"ticker": "AAPL_US_EQ", "action": "BUY", "target_allocation_pct": 8.0,
     "conviction": 72, "primary_strategy": "momentum",
     "reasoning": "Strong momentum: above 50-day MA, RSI in sweet spot, positive MACD."},
    {"ticker": "MSFT_US_EQ", "action": "BUY", "target_allocation_pct": 6.5,
     "conviction": 65, "primary_strategy": "mean_reversion",
     "reasoning": "Trading near lower Bollinger Band with strong fundamentals."},
    {"ticker": "GOOGL_US_EQ", "action": "BUY", "target_allocation_pct": 7.0,
     "conviction": 60, "primary_strategy": "factor",
     "reasoning": "High factor composite score with strong quality metrics."},
]
sample_portfolio = json.dumps({"cash": 10000, "total_value": 10000, "positions": []})

# Placeholders for Order Manager / Journal (populated when Moderation + Risk sections run)
panel_results = {}
risk_verdicts = {}
exec_results = {}

print("Quick Setup complete: fetcher, df, macro, analysis, stocks_data, sub_results, test_trades ready.")

Quick Setup complete: fetcher, df, macro, analysis, stocks_data, sub_results, test_trades ready.


---
## 1. Configuration & Environment Variables

In [3]:
from src.utils.config import get_settings, Settings

settings = get_settings()

print("=== Trading Settings ===")
print(f"  Mode            : {settings.trading.get('mode', 'N/A')}")
print(f"  Base URL        : {settings.t212_base_url}")
print(f"  Cycle hours     : {settings.cycle_hours}")
print(f"  Cycle times UTC : {settings.cycle_times_utc}")
print(f"  Max positions   : {settings.max_positions}")
print(f"  Min position %  : {settings.min_position_pct}")
print(f"  Max position %  : {settings.max_position_pct}")
print(f"  Cash floor %    : {settings.cash_floor_pct}")
print(f"  Benchmark       : {settings.benchmark_ticker}")

print("\n=== Risk Settings ===")
print(f"  Max single stock : {settings.max_single_stock_pct}%")
print(f"  Max sector       : {settings.max_sector_pct}%")
print(f"  Max correlation  : {settings.max_correlation}")
print(f"  Cautious DD %    : {settings.cautious_drawdown_pct}")
print(f"  Halt DD %        : {settings.halt_drawdown_pct}")
print(f"  VIX high/extreme : {settings.vix_high} / {settings.vix_extreme}")

print("\n=== Strategy Settings ===")
print(f"  Momentum weight      : {settings.momentum_weight}")
print(f"  Mean reversion weight : {settings.mean_reversion_weight}")
print(f"  Factor weight        : {settings.factor_weight}")
print(f"  Min conviction       : {settings.min_conviction}")

print("\n=== Models ===")
print(f"  Strategy   : {settings.strategy_model}")
print(f"  Moderator 1: {settings.moderator_1_model}")
print(f"  Moderator 2: {settings.moderator_2_model}")

print("\n=== Cost Limits ===")
print(f"  Anthropic daily : \u00a3{settings.anthropic_daily_gbp}")
print(f"  OpenAI daily    : \u00a3{settings.openai_daily_gbp}")
print(f"  Google daily    : \u00a3{settings.google_daily_gbp}")
print(f"  Monthly total   : \u00a3{settings.total_monthly_gbp}")

print("\n=== Config: PASS ===")

=== Trading Settings ===
  Mode            : active
  Base URL        : https://demo.trading212.com/api/v0
  Cycle hours     : 4
  Cycle times UTC : ['08:00', '12:00', '16:00']
  Max positions   : 15
  Min position %  : 2.0
  Max position %  : 15.0
  Cash floor %    : 10.0
  Benchmark       : ^GSPC

=== Risk Settings ===
  Max single stock : 15.0%
  Max sector       : 35.0%
  Max correlation  : 0.7
  Cautious DD %    : 5.0
  Halt DD %        : 15.0
  VIX high/extreme : 25.0 / 35.0

=== Strategy Settings ===
  Momentum weight      : 0.35
  Mean reversion weight : 0.3
  Factor weight        : 0.35
  Min conviction       : 75

=== Models ===
  Strategy   : claude-sonnet-4-5-20250929
  Moderator 1: gpt-4o
  Moderator 2: gemini-2.5-flash

=== Cost Limits ===
  Anthropic daily : £1.0
  OpenAI daily    : £0.75
  Google daily    : £0.5
  Monthly total   : £50.0

=== Config: PASS ===


In [4]:
# Verify all required environment variables are set (mask values)
env_keys = [
    "T212_API_KEY", "T212_API_SECRET",
    "ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GOOGLE_AI_API_KEY",
    "FINNHUB_API_KEY", "ALPHA_VANTAGE_API_KEY",
]

all_ok = True
for key in env_keys:
    val = os.getenv(key)
    if val:
        masked = val[:4] + "..." + val[-4:] if len(val) > 8 else "****"
        print(f"  {key:25s} = {masked}")
    else:
        print(f"  {key:25s} = MISSING")
        all_ok = False

env_status = "PASS" if all_ok else "WARN (some keys missing)"
print(f"\n=== Env Vars: {env_status} ===")

  T212_API_KEY              = 3264...UrtD
  T212_API_SECRET           = zolx...5VFA
  ANTHROPIC_API_KEY         = sk-a...dQAA
  OPENAI_API_KEY            = sk-p...ECkA
  GOOGLE_AI_API_KEY         = AIza...w9JE
  FINNHUB_API_KEY           = d6jg...nbe0
  ALPHA_VANTAGE_API_KEY     = HKZ4...H7YP

=== Env Vars: PASS ===


---
## 2. Database & Models

In [5]:
from sqlalchemy import inspect as sa_inspect, text
from src.data.database import engine, get_session
from src.data.models import Base

Base.metadata.create_all(engine)

inspector = sa_inspect(engine)
table_names = inspector.get_table_names()
print(f"Database : {engine.url}")
print(f"Tables ({len(table_names)}):")

session = get_session()
try:
    for t in sorted(table_names):
        count = session.execute(text(f'SELECT COUNT(*) FROM "{t}"')).scalar()
        print(f"  {t:35s} {count:>6} rows")
finally:
    session.close()

print(f"\n=== Database: PASS ===")

Database : sqlite:////Users/Kayvan/Library/Mobile Documents/com~apple~CloudDocs/AI_Projects/Investment-agent/data/investment_agent.db
Tables (17):
  alembic_version                          1 rows
  api_logs                              1632 rows
  cost_logs                              225 rows
  instruments                          16617 rows
  market_data_cache                      456 rows
  moderation_logs                        294 rows
  news_sentiment_cache                     1 rows
  notification_logs                       11 rows
  opportunity_queue                        1 rows
  opportunity_score_snapshots            108 rows
  orders                                 105 rows
  performance_metrics                      2 rows
  portfolio_snapshots                     26 rows
  risk_decisions                          92 rows
  strategy_decisions                     178 rows
  system_state                             1 rows
  trade_outcomes                          50 rows

==

---
## 3. State Machine

In [6]:
from src.orchestrator.state_machine import StateMachine

sm = StateMachine()
state = sm.get_state()

print("=== System State ===")
for k, v in state.items():
    print(f"  {k:25s} : {v}")

assert state["state"] in {"ACTIVE", "CAUTIOUS", "HALTED"}, f"Invalid state: {state['state']}"
print(f"\nCurrent state : {state['state']}")
print(f"Is paused     : {state.get('paused', False)}")
print(f"\n=== State Machine: PASS ===")

=== System State ===
  state                     : ACTIVE
  peak_portfolio_value      : 10000.0
  current_drawdown_pct      : 0.00819999999999709
  last_cycle_at             : 2026-03-04 22:24:21.437542
  daily_loss_halt_until     : None
  paused                    : False
  notes                     : None

Current state : ACTIVE
Is paused     : False

=== State Machine: PASS ===


---
## 4. Cost Tracker & Budget Status

In [7]:
from src.utils.cost_tracker import (
    Provider, get_budget_status, get_degradation_level,
    get_cost_summary, get_daily_spend, get_monthly_spend,
)

print("=== Budget Status ===")
for provider in Provider:
    bs = get_budget_status(provider.value)
    print(f"\n  {provider.value.upper()}:")
    print(f"    Daily  : \u00a3{bs.daily_spent_gbp:.4f} / \u00a3{bs.daily_limit_gbp:.2f}  ({bs.daily_pct_used:.1f}%)")
    print(f"    Monthly: \u00a3{bs.monthly_spent_gbp:.4f} / \u00a3{bs.monthly_limit_gbp:.2f}  ({bs.monthly_pct_used:.1f}%)")
    print(f"    Over daily: {bs.is_over_daily}  |  Over monthly: {bs.is_over_monthly}")

deg = get_degradation_level()
print(f"\nDegradation level : {deg.value}")
print(f"Total today       : \u00a3{get_daily_spend():.4f}")
print(f"Total this month  : \u00a3{get_monthly_spend():.4f}")

summary = get_cost_summary(days=7)
print(f"\n7-day summary: {json.dumps(summary, indent=2)}")

print(f"\n=== Cost Tracker: PASS ===")

=== Budget Status ===

  ANTHROPIC:
    Daily  : £0.0000 / £1.00  (0.0%)
    Monthly: £1.3507 / £50.00  (2.7%)
    Over daily: False  |  Over monthly: False

  OPENAI:
    Daily  : £0.0000 / £0.75  (0.0%)
    Monthly: £1.3507 / £50.00  (2.7%)
    Over daily: False  |  Over monthly: False

  GOOGLE:
    Daily  : £0.0000 / £0.50  (0.0%)
    Monthly: £1.3507 / £50.00  (2.7%)
    Over daily: False  |  Over monthly: False

Degradation level : full
Total today       : £0.0000
Total this month  : £1.3507

7-day summary: {
  "anthropic": 1.204716,
  "google": 0.018628,
  "openai": 0.47934,
  "total": 1.702684
}

=== Cost Tracker: PASS ===


---
## 5. Market Data — yfinance (OHLCV + Indicators)

In [8]:
from src.agents.market_data.data_fetcher import DataFetcher

fetcher = DataFetcher()

TEST_TICKER = "AAPL"
print(f"Fetching 1-year OHLCV for {TEST_TICKER}...")
df = fetcher.get_ohlcv(TEST_TICKER, period="1y")
assert not df.empty, "OHLCV data is empty!"
print(f"  Rows    : {len(df)}")
print(f"  Columns : {list(df.columns)}")
print(f"  Range   : {df.index[0].date()} to {df.index[-1].date()}")
print(f"  Latest close: ${float(df['Close'].iloc[-1]):.2f}")

print(f"\n=== yfinance OHLCV: PASS ===")

Fetching 1-year OHLCV for AAPL...
  Rows    : 252
  Columns : ['Close', 'High', 'Low', 'Open', 'Volume']
  Range   : 2025-03-06 to 2026-03-06
  Latest close: $256.60

=== yfinance OHLCV: PASS ===


In [9]:
from src.agents.market_data.indicators import calculate_indicators, calculate_relative_strength

print(f"Calculating indicators for {TEST_TICKER}...")
indicators = calculate_indicators(df)

assert "error" not in indicators, f"Indicator error: {indicators.get('error')}"

# Verify only the 8 expected fields are present (see docs/DATA_RATIONALE.md)
EXPECTED_INDICATOR_KEYS = {
    "current_price", "rsi_14", "macd_histogram",
    "macd_bullish_crossover", "macd_bearish_crossover",
    "below_lower_bb", "above_50ma", "ma_20",
}
actual_keys = set(indicators.keys())
assert actual_keys == EXPECTED_INDICATOR_KEYS, (
    f"Unexpected indicator keys.\n"
    f"  Extra: {actual_keys - EXPECTED_INDICATOR_KEYS}\n"
    f"  Missing: {EXPECTED_INDICATOR_KEYS - actual_keys}"
)

print(f"\n=== Technical Indicators ({len(indicators)} fields) ===")
for k, v in indicators.items():
    if isinstance(v, float):
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

print(f"\nFetching benchmark data...")
bench_df = fetcher.get_benchmark_data()
assert not bench_df.empty, "Benchmark data is empty!"
rs = calculate_relative_strength(df, bench_df)
print(f"  {TEST_TICKER} relative strength (6m) vs S&P 500: {rs:.4f}")

print(f"\n=== Indicators: PASS ===")

Calculating indicators for AAPL...

=== Technical Indicators (8 fields) ===
  current_price             : 256.6000
  rsi_14                    : 40.0927
  macd_histogram            : -1.2043
  macd_bullish_crossover    : False
  macd_bearish_crossover    : False
  below_lower_bb            : False
  above_50ma                : False
  ma_20                     : 266.5060

Fetching benchmark data...
  AAPL relative strength (6m) vs S&P 500: 1.0294

=== Indicators: PASS ===


---
## 6. Fundamentals (yfinance)

In [10]:
from src.agents.market_data.fundamentals import get_fundamentals

print(f"Fetching fundamentals for {TEST_TICKER}...")
fundamentals = get_fundamentals(TEST_TICKER)

assert "error" not in fundamentals, f"Fundamentals error: {fundamentals.get('error')}"

# Verify expected fields are present (see docs/DATA_RATIONALE.md)
EXPECTED_FUNDAMENTAL_KEYS = {
    "trailing_pe", "pb_ratio", "roe", "profit_margin",
    "debt_equity", "earnings_growth", "earnings_momentum_qoq",
    "sector", "market_cap", "industry", "business_summary",
}
actual_keys = set(fundamentals.keys())
assert actual_keys == EXPECTED_FUNDAMENTAL_KEYS, (
    f"Unexpected fundamental keys.\n"
    f"  Extra: {actual_keys - EXPECTED_FUNDAMENTAL_KEYS}\n"
    f"  Missing: {EXPECTED_FUNDAMENTAL_KEYS - actual_keys}"
)

print(f"\n=== Fundamentals ({len(fundamentals)} fields) ===")
for k, v in fundamentals.items():
    if isinstance(v, float) and v is not None:
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

print(f"\n=== Fundamentals: PASS ===")

Fetching fundamentals for AAPL...

=== Fundamentals (11 fields) ===
  trailing_pe               : 32.4431
  pb_ratio                  : 42.7851
  roe                       : 1.5202
  profit_margin             : 0.2704
  debt_equity               : 102.6300
  earnings_growth           : 0.1830
  earnings_momentum_qoq     : 0.5327
  sector                    : Technology
  industry                  : Consumer Electronics
  market_cap                : 3771855273984.0000
  business_summary          : Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates

---
## 7. Macro Data & Market Regime

In [11]:
print("Fetching macro data (VIX, S&P 500 vs 200-day MA)...")
macro = fetcher.get_macro_data()

print(f"\n=== Macro Data ===")
for k, v in macro.items():
    if isinstance(v, float):
        print(f"  {k:25s} : {v:.4f}")
    else:
        print(f"  {k:25s} : {v}")

# Verify yield spread fields were removed
assert "yield_spread_10y_short" not in macro, "yield_spread should be removed (see DATA_RATIONALE.md)"
assert "ten_year_yield" not in macro, "ten_year_yield should be removed"
assert "short_yield" not in macro, "short_yield should be removed"

print(f"\nMarket regime : {macro.get('market_regime', 'UNKNOWN')}")
print(f"VIX           : {macro.get('vix', 'N/A')}")

print(f"\n=== Macro Data: PASS ===")

Fetching macro data (VIX, S&P 500 vs 200-day MA)...

=== Macro Data ===
  vix                       : 26.1500
  sp500_price               : 6756.7402
  sp500_200ma               : 6582.6121
  sp500_above_200ma         : True
  sp500_pct_above_200ma     : 2.6453
  market_regime             : SIDEWAYS
  macro_intelligence        : {'enabled': True, 'fetched_at': '2026-03-06T16:31:48.740852+00:00', 'sector_trends': {}, 'sector_summary': 'Sector data unavailable.', 'economic_highlights': "- Iran spent years fostering proxies in Iraq. Now, many aren’t eager to join the war - Reuters (Reuters)\n- United Airlines CEO: Fuel spike will hit results, but travel demand hasn't taken 'even a tiny step back' (CNBC)\n- Sri Lanka moving 208 rescued Iranian ship crew to naval camp, sources say - Reuters (Reuters)\n- US airlines no longer hedge fuel costs. That could hurt margins if Iran conflict lingers - Reuters (Reuters)\n- White House to press defense firms to boost production as US strikes on Iran d

---
## 7.5 Macro Intelligence (Sector Performance + Economic Headlines)

**Sources:** Alpha Vantage SECTOR (1 call), Finnhub /news (1 call). Cached 4h. Enables "sector headwind — defer buy" in committee decisions.

In [13]:
# Ensure macro is available (from Section 7 or Quick Setup)
if 'macro' not in dir() or not macro:
    macro = fetcher.get_macro_data()

macro_intel = fetcher.get_macro_intelligence_cached()

print("=== Macro Intelligence ===")
print(f"  Enabled           : {macro_intel.get('enabled', False)}")
print(f"  Sector summary    : {macro_intel.get('sector_summary', 'N/A')[:200]}...")
print(f"  Economic highlights: {macro_intel.get('economic_highlights', 'N/A')[:200]}...")
print(f"  Earnings season   : {macro_intel.get('earnings_season_flag', False)}")

# Test sector headwind for Technology
from src.agents.market_data.macro_intelligence import get_sector_headwind
hw = get_sector_headwind(macro_intel, "Technology")
print(f"  Sector headwind (Technology): {hw or 'None'}")

if macro_intel.get("errors"):
    print(f"  Errors: {macro_intel['errors']}")

macro_intel_status = "PASS" if macro_intel.get("enabled") or not macro_intel.get("errors") else "WARN"
print(f"\n=== Macro Intelligence: {macro_intel_status} ===")

=== Macro Intelligence ===
  Enabled           : True
  Sector summary    : Sector data unavailable....
  Economic highlights: - Iran spent years fostering proxies in Iraq. Now, many aren’t eager to join the war - Reuters (Reuters)
- United Airlines CEO: Fuel spike will hit results, but travel demand hasn't taken 'even a tiny...
  Earnings season   : False
  Sector headwind (Technology): None

=== Macro Intelligence: PASS ===


---
## 8. Finnhub API (Analyst Recommendations + Insider Sentiment)

In [14]:
print(f"Testing Finnhub API for {TEST_TICKER}...")

try:
    finnhub = fetcher.finnhub

    print("\n--- Analyst Recommendations ---")
    recs = finnhub.get_analyst_recommendations(TEST_TICKER)
    for k, v in recs.items():
        print(f"  {k:20s} : {v}")

    print("\n--- Insider Sentiment ---")
    insider = finnhub.get_insider_sentiment(TEST_TICKER)
    for k, v in insider.items():
        print(f"  {k:20s} : {v}")

    print("\n--- Combined Analyst Data ---")
    analyst_data = finnhub.get_analyst_data(TEST_TICKER)
    print(f"  Keys: {list(analyst_data.keys())}")

    finnhub_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    finnhub_status = f"FAIL ({e})"

print(f"\n=== Finnhub: {finnhub_status} ===")

Testing Finnhub API for AAPL...

--- Analyst Recommendations ---
  symbol               : AAPL
  unavailable          : False
  period               : 2026-03-01
  strong_buy           : 14
  buy                  : 22
  hold                 : 16
  sell                 : 2
  strong_sell          : 0
  total_analysts       : 54
  consensus            : BUY

--- Insider Sentiment ---
  symbol               : AAPL
  unavailable          : False
  mspr                 : 25.651577
  change               : 6732
  month                : 2
  year                 : 2026

--- Combined Analyst Data ---
  Keys: ['analyst_recommendations', 'insider_sentiment']

=== Finnhub: PASS ===


---
## 9. Alpha Vantage API (News Sentiment)

**Note:** Free tier = 25 requests/day. This section uses 2 calls.

In [15]:
print("Testing Alpha Vantage news sentiment API...\n")

try:
    av = fetcher.alpha_vantage

    print("--- Broad Market Sentiment ---")
    broad = av.get_broad_market_sentiment()
    if "error" in broad:
        print(f"  Error: {broad['error']}")
    else:
        print(f"  Total articles   : {broad.get('total_articles', 0)}")
        print(f"  Avg sentiment    : {broad.get('average_sentiment', 0):.4f}")
        print(f"  Bullish articles : {broad.get('bullish_articles', 0)}")
        print(f"  Bearish articles : {broad.get('bearish_articles', 0)}")
        broad_articles = broad.get("articles", [])
        if broad_articles:
            print(f"  Top headlines:")
            from src.agents.market_data.alpha_vantage_client import AlphaVantageClient
            for line in AlphaVantageClient._summarize_articles(broad_articles, max_articles=5).split('\n'):
                print(f"    {line}")

    print(f"\n--- Ticker Sentiment ({TEST_TICKER}) ---")
    ticker_sent = av.get_ticker_news_summary(tickers=TEST_TICKER, limit=10)
    if "error" in ticker_sent:
        print(f"  Error: {ticker_sent['error']}")
    else:
        print(f"  Total articles   : {ticker_sent.get('total_articles', 0)}")
        print(f"  Avg sentiment    : {ticker_sent.get('average_sentiment', 0):.4f}")
        print(f"  Bullish articles : {ticker_sent.get('bullish_articles', 0)}")
        print(f"  Bearish articles : {ticker_sent.get('bearish_articles', 0)}")
        summary_text = ticker_sent.get('top_articles_summary', '')
        if summary_text and summary_text != "No recent articles found.":
            print(f"  Top 10 articles (fed to LLM):")
            for line in summary_text.split('\n'):
                print(f"    {line}")
        else:
            print(f"  WARNING: No articles returned — LLM will not receive news context")

    # Build the same news_sentiment string the orchestrator builds (for use in cell 12)
    news_parts = []
    if ticker_sent and "error" not in ticker_sent:
        news_parts.append(f"### Ticker News ({ticker_sent.get('tickers_queried', 'N/A')})")
        news_parts.append(f"Articles: {ticker_sent.get('total_articles', 0)} | "
                          f"Avg sentiment: {ticker_sent.get('average_sentiment', 0):.4f} | "
                          f"Bullish: {ticker_sent.get('bullish_articles', 0)} | "
                          f"Bearish: {ticker_sent.get('bearish_articles', 0)}")
        summary = ticker_sent.get("top_articles_summary", "")
        if summary:
            news_parts.append(summary)
    if broad and "error" not in broad:
        news_parts.append(f"\n### Broad Market Sentiment")
        news_parts.append(f"Articles: {broad.get('total_articles', 0)} | "
                          f"Avg sentiment: {broad.get('average_sentiment', 0):.4f} | "
                          f"Bullish: {broad.get('bullish_articles', 0)} | "
                          f"Bearish: {broad.get('bearish_articles', 0)}")
        broad_arts = broad.get("articles", [])
        if broad_arts:
            news_parts.append(AlphaVantageClient._summarize_articles(broad_arts, max_articles=10))

    diag_news_sentiment = "\n".join(news_parts)[:3000] if news_parts else "News sentiment data unavailable."

    print(f"\n--- News Sentiment String for LLM (preview) ---")
    print(diag_news_sentiment[:500])
    if len(diag_news_sentiment) > 500:
        print(f"  ... ({len(diag_news_sentiment)} chars total)")

    av_status = "PASS"
except Exception as e:
    print(f"  ERROR: {e}")
    diag_news_sentiment = "News sentiment data unavailable."
    av_status = f"FAIL ({e})"

print(f"\n=== Alpha Vantage: {av_status} ===")

Testing Alpha Vantage news sentiment API...

--- Broad Market Sentiment ---
  Total articles   : 17
  Avg sentiment    : 0.1186
  Bullish articles : 8
  Bearish articles : 3
  Top headlines:
    [Neutral -0.073] Stock Market Today: Dow Jones, Nasdaq 100 Futures Rise After Trump's State Of The Union Address—Nvid (Benzinga)
    [Somewhat-Bearish -0.349] Kforce, Insperity, PAR Technology, Pure Storage, and Xerox Stocks Trade Down, What You Need To Know (TradingView)
    [Bullish +0.381] Tim S.A. Earnings Call Highlights Growth And Challenges (The Globe and Mail)
    [Neutral +0.014] Stock Market Today: Dow Futures Gain, Nasdaq Drops As Trump Signs Bill To End Partial Shutdown—AMD,  (Sahm)
    [Neutral -0.072] Intuit (INTU) stock price slides late Tuesday as IRS tax season opens and director pay filing lands (TechStock²)

--- Ticker Sentiment (AAPL) ---
  Total articles   : 10
  Avg sentiment    : 0.1801
  Bullish articles : 5
  Bearish articles : 0
  Top 10 articles (fed to LLM):
    [Neu

---
## 10. Full Stock Analysis (Combined Data Pipeline)

In [16]:
print(f"Running full stock analysis for {TEST_TICKER}...")
print("(Combines: OHLCV + indicators + relative strength + fundamentals + Finnhub)\n")

analysis = fetcher.get_stock_analysis(TEST_TICKER)

print(f"Top-level keys: {list(analysis.keys())}")

ind_keys = list(analysis.get("indicators", {}).keys())
fund_keys = list(analysis.get("fundamentals", {}).keys())
print(f"\nIndicators ({len(ind_keys)})  : {ind_keys}")
print(f"Fundamentals ({len(fund_keys)}): {fund_keys}")
print(f"Relative strength 6m : {analysis.get('relative_strength_6m', 'N/A')}")
print(f"Analyst data keys    : {list(analysis.get('analyst_data', {}).keys())}")

assert "indicators" in analysis
assert "fundamentals" in analysis

# Verify cleaned data shapes
ind = analysis.get("indicators", {})
if "error" not in ind:
    assert len(ind) == 8, f"Expected 8 indicator fields, got {len(ind)}: {list(ind.keys())}"
fund = analysis.get("fundamentals", {})
if "error" not in fund:
    assert len(fund) == 11, f"Expected 11 fundamental fields, got {len(fund)}: {list(fund.keys())}"

print(f"\n=== Full Stock Analysis: PASS ===")

Running full stock analysis for AAPL...
(Combines: OHLCV + indicators + relative strength + fundamentals + Finnhub)

Top-level keys: ['ticker', 'timestamp', 'indicators', 'relative_strength_6m', 'fundamentals', 'analyst_data']

Indicators (8)  : ['current_price', 'rsi_14', 'macd_histogram', 'macd_bullish_crossover', 'macd_bearish_crossover', 'below_lower_bb', 'above_50ma', 'ma_20']
Fundamentals (11): ['trailing_pe', 'pb_ratio', 'roe', 'profit_margin', 'debt_equity', 'earnings_growth', 'earnings_momentum_qoq', 'sector', 'industry', 'market_cap', 'business_summary']
Relative strength 6m : 1.029587923926499
Analyst data keys    : ['analyst_recommendations', 'insider_sentiment']


AssertionError: Expected 9 fundamental fields, got 11: ['trailing_pe', 'pb_ratio', 'roe', 'profit_margin', 'debt_equity', 'earnings_growth', 'earnings_momentum_qoq', 'sector', 'industry', 'market_cap', 'business_summary']

---
## 11. Sub-Strategies (Momentum / Mean Reversion / Factor)

In [ ]:
from src.agents.strategy.momentum import evaluate_momentum
from src.agents.strategy.mean_reversion import evaluate_mean_reversion
from src.agents.strategy.factor import calculate_factor_score, rank_by_factor

ind = analysis.get("indicators", {})
fund = analysis.get("fundamentals", {})
rs_val = analysis.get("relative_strength_6m")

print("=== Momentum Strategy ===")
mom = evaluate_momentum(TEST_TICKER, ind, rs_val, current_holding=False)
print(f"  Action    : {mom.action}")
print(f"  Score     : {mom.score:.0f}/100")
print(f"  Reasoning : {mom.reasoning}")

print("\n=== Mean Reversion Strategy ===")
mr = evaluate_mean_reversion(TEST_TICKER, ind, fund, current_holding=False)
print(f"  Action    : {mr.action}")
print(f"  Score     : {mr.score:.0f}/100")
print(f"  Reasoning : {mr.reasoning}")

print("\n=== Factor Strategy ===")
factor = calculate_factor_score(TEST_TICKER, fund, ind, rs_val)
print(f"  Composite : {factor.composite_score:.0f}/100")
print(f"  Value     : {factor.value_score:.0f}")
print(f"  Quality   : {factor.quality_score:.0f}")
print(f"  Momentum  : {factor.momentum_score:.0f}")
print(f"  Reasoning : {factor.reasoning}")

print(f"\n=== Sub-Strategies: PASS ===")

In [ ]:
from src.agents.strategy.engine import StrategyEngine

MULTI_TICKERS = ["AAPL", "MSFT", "GOOGL"]
print(f"Testing sub-strategies on: {MULTI_TICKERS}")
print("(Fetching data — may take a moment...)\n")

stocks_data = []
for ticker in MULTI_TICKERS:
    try:
        data = fetcher.get_stock_analysis(ticker)
        data["ticker"] = ticker
        stocks_data.append(data)
        print(f"  {ticker}: OK (price=${data.get('indicators', {}).get('current_price', 'N/A')})")
    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")
        stocks_data.append({"ticker": ticker, "indicators": {}, "fundamentals": {}})

engine = StrategyEngine()
sub_results = engine.run_sub_strategies(stocks_data, existing_positions=set())

print(f"\nMomentum signals : {len(sub_results['momentum'])}")
for s in sub_results["momentum"]:
    print(f"  {s.ticker}: {s.action} (score={s.score:.0f})")

print(f"\nMean reversion signals : {len(sub_results['mean_reversion'])}")
for s in sub_results["mean_reversion"]:
    print(f"  {s.ticker}: {s.action} (score={s.score:.0f})")

print(f"\nTop factor scores : {len(sub_results.get('top_factor', []))}")
for s in sub_results.get("top_factor", []):
    print(f"  {s.ticker}: composite={s.composite_score:.0f} (V={s.value_score:.0f} Q={s.quality_score:.0f} M={s.momentum_score:.0f})")

print(f"\n=== Multi-Ticker Sub-Strategies: PASS ===")

---
## 12. Claude Strategy Synthesis (Anthropic API)

**Cost:** ~$0.01-0.03 per call.

In [ ]:
print("Testing Claude strategy synthesis...")
print("(This makes a real Anthropic API call)\n")

# --- Fetch per-ticker news for ALL MULTI_TICKERS (1 API call) ---
# Cell 9 only fetched news for TEST_TICKER ("AAPL").
# The orchestrator fetches news for all top tickers; replicate that here.
from src.agents.market_data.alpha_vantage_client import AlphaVantageClient
from src.agents.market_data.data_fetcher import DataFetcher

_multi_tickers_str = ",".join(MULTI_TICKERS)
_per_ticker_news: dict[str, str] = {}
_av_ticker_sentiment: dict[str, any] = {}
_av_all_articles: list[dict] = []

try:
    _raw = fetcher.alpha_vantage.get_market_news_sentiment(
        tickers=_multi_tickers_str, sort="RELEVANCE", limit=30,
    )
    if "error" not in _raw:
        _av_all_articles = _raw.get("articles", [])
        _av_ticker_sentiment = {
            "tickers_queried": _multi_tickers_str,
            "total_articles": _raw.get("total_articles", 0),
            "average_sentiment": _raw.get("average_sentiment", 0),
            "bullish_articles": _raw.get("bullish_articles", 0),
            "bearish_articles": _raw.get("bearish_articles", 0),
            "neutral_articles": _raw.get("neutral_articles", 0),
        }
        # Extract per-ticker news from raw articles
        _per_ticker_news = DataFetcher.extract_per_ticker_news(_av_all_articles, MULTI_TICKERS)
        print(f"Fetched {len(_av_all_articles)} articles for {_multi_tickers_str}")
        for _t, _txt in _per_ticker_news.items():
            print(f"  {_t}: {'has news (' + str(len(_txt)) + ' chars)' if _txt else 'no matching articles'}")
    else:
        print(f"Alpha Vantage error: {_raw.get('error')}")
except Exception as e:
    print(f"Alpha Vantage unavailable: {e}")

# --- Build news_sentiment string matching orchestrator format ---
_news_parts: list[str] = []

# Per-ticker section (the critical part that was missing)
if _per_ticker_news:
    _news_parts.append("### Per-Ticker News Sentiment")
    for _yf_t, _news_text in _per_ticker_news.items():
        if _news_text:
            _news_parts.append(f"\n**{_yf_t}**:\n{_news_text}")

# Aggregate ticker stats
if _av_ticker_sentiment:
    _news_parts.append(f"\n### Aggregate Ticker News ({_av_ticker_sentiment.get('tickers_queried', 'N/A')})")
    _news_parts.append(f"Articles: {_av_ticker_sentiment.get('total_articles', 0)} | "
                        f"Avg sentiment: {_av_ticker_sentiment.get('average_sentiment', 0):.4f} | "
                        f"Bullish: {_av_ticker_sentiment.get('bullish_articles', 0)} | "
                        f"Bearish: {_av_ticker_sentiment.get('bearish_articles', 0)}")

# Broad market sentiment from cell 9 (if available)
_broad = globals().get("broad", {})
if _broad and "error" not in _broad:
    _news_parts.append(f"\n### Broad Market Sentiment")
    _news_parts.append(f"Articles: {_broad.get('total_articles', 0)} | "
                        f"Avg sentiment: {_broad.get('average_sentiment', 0):.4f} | "
                        f"Bullish: {_broad.get('bullish_articles', 0)} | "
                        f"Bearish: {_broad.get('bearish_articles', 0)}")
    _broad_arts = _broad.get("articles", [])
    if _broad_arts:
        _news_parts.append(AlphaVantageClient._summarize_articles(_broad_arts, max_articles=10))

_news_for_claude = "\n".join(_news_parts)[:3000] if _news_parts else "News sentiment data unavailable."

print(f"\nNews sentiment source: {'Per-ticker + aggregate Alpha Vantage data' if _per_ticker_news else 'Aggregate only (no per-ticker matches)'}")
print(f"News sentiment length: {len(_news_for_claude)} chars")
print(f"\n--- Preview (first 600 chars) ---")
print(_news_for_claude[:600])
if len(_news_for_claude) > 600:
    print(f"  ... ({len(_news_for_claude)} chars total)")

# --- Claude synthesis ---
print("\n--- Calling Claude ---")
try:
    portfolio_state = json.dumps({
        "cash": 10000.0, "total_value": 10000.0,
        "invested": 0.0, "positions": [], "num_positions": 0,
    })

    from src.orchestrator.main import Orchestrator
    _company_profiles = Orchestrator._build_company_profiles(stocks_data, MULTI_TICKERS)
    synth_result = engine.synthesize_with_claude(
        sub_strategy_results=sub_results,
        portfolio_state=portfolio_state,
        market_regime=macro.get("market_regime", "SIDEWAYS"),
        analyst_data="{}",
        news_sentiment=_news_for_claude,
        company_profiles=_company_profiles,
        system_state="ACTIVE",
        vix=macro.get("vix"),
        cash_pct=100.0,
        num_positions=0,
        cycle_id="diag_test",
    )

    if "error" in synth_result and not synth_result.get("decisions"):
        print(f"  Strategy error: {synth_result['error']}")
        claude_status = f"WARN ({synth_result['error']})"
    else:
        decisions = synth_result.get("decisions", [])
        print(f"  Market assessment : {synth_result.get('market_assessment', 'N/A')[:120]}")
        print(f"  Decisions         : {len(decisions)}")
        for d in decisions[:5]:
            print(f"    {d.get('ticker')}: {d.get('action')} @ {d.get('target_allocation_pct', 0):.1f}%  conviction={d.get('conviction', 0)}")
            news_sum = d.get('news_sentiment_summary', '')
            if news_sum:
                print(f"      News context: {news_sum}")
        claude_status = "PASS"

except Exception as e:
    print(f"  ERROR: {e}")
    claude_status = f"FAIL ({e})"

print(f"\n=== Claude Synthesis: {claude_status} ===")

---
## 13. Moderation Panel (GPT-4o + Gemini)

**Cost:** ~$0.006 (OpenAI + Google combined).

In [ ]:
from src.agents.moderation import openai_mod, gemini_mod
from src.agents.moderation.context import format_market_context

# --- Build test trades from Claude synthesis decisions or defaults ---
# In production, the orchestrator loops through ALL Claude decisions.
# We replicate that by testing each MULTI_TICKER through the moderators.
_claude_decisions = globals().get("synth_result", {}).get("decisions", [])

if _claude_decisions and len(_claude_decisions) >= 2:
    test_trades = _claude_decisions[:len(MULTI_TICKERS)]
    print(f"Using {len(test_trades)} decisions from Claude synthesis (cell 12)")
else:
    test_trades = [
        {"ticker": "AAPL_US_EQ", "action": "BUY", "target_allocation_pct": 8.0,
         "conviction": 72, "primary_strategy": "momentum",
         "reasoning": "Strong momentum: above 50-day MA, RSI in sweet spot, positive MACD."},
        {"ticker": "MSFT_US_EQ", "action": "BUY", "target_allocation_pct": 6.5,
         "conviction": 65, "primary_strategy": "mean_reversion",
         "reasoning": "Trading near lower Bollinger Band with strong fundamentals."},
        {"ticker": "GOOGL_US_EQ", "action": "BUY", "target_allocation_pct": 7.0,
         "conviction": 60, "primary_strategy": "factor",
         "reasoning": "High factor composite score with strong quality metrics."},
    ]
    print(f"Using {len(test_trades)} default test trades")

sample_portfolio = json.dumps({"cash": 10000, "total_value": 10000, "positions": []})


def _build_diag_market_context(ticker: str) -> dict:
    """Build per-ticker market context using data from earlier cells (mirrors orchestrator._build_market_context)."""
    yf_t = ticker.replace("_US_EQ", "").replace("_UK_EQ", "")
    stock = next((s for s in stocks_data if s.get("ticker") in (yf_t, ticker)), {})
    _ind = stock.get("indicators", {})
    _fund = stock.get("fundamentals", {})

    _mom = _mr = _fac = None
    for s in sub_results.get("momentum", []):
        if s.ticker in (yf_t, ticker):
            _mom = {"action": s.action, "score": s.score, "reasoning": s.reasoning}
            break
    for s in sub_results.get("mean_reversion", []):
        if s.ticker in (yf_t, ticker):
            _mr = {"action": s.action, "score": s.score, "reasoning": s.reasoning}
            break
    for s in sub_results.get("factor", []):
        if s.ticker in (yf_t, ticker):
            _fac = {"composite_score": s.composite_score, "value_score": s.value_score,
                     "quality_score": s.quality_score, "momentum_score": s.momentum_score,
                     "reasoning": s.reasoning}
            break

    _ticker_news = globals().get("_per_ticker_news", {}).get(yf_t, "")
    macro_intel = macro.get("macro_intelligence", {})
    sector = _fund.get("sector", "Unknown")
    from src.agents.market_data.macro_intelligence import get_sector_headwind
    sector_headwind = get_sector_headwind(macro_intel, sector) if macro_intel.get("enabled") else None
    return {
        "indicators": _ind,
        "fundamentals": _fund,
        "macro": {
            "vix": macro.get("vix"), "market_regime": macro.get("market_regime", "SIDEWAYS"),
            "sp500_above_200ma": macro.get("sp500_above_200ma"),
            "sector_headwind": sector_headwind,
            "economic_highlights": macro_intel.get("economic_highlights", ""),
            "sector_summary": macro_intel.get("sector_summary", ""),
        },
        "sub_strategies": {"momentum": _mom, "mean_reversion": _mr, "factor": _fac},
        "analyst_data": {},
        "news_sentiment": _ticker_news or globals().get("diag_news_sentiment", ""),
    }


# --- Test GPT-4o moderator for each trade ---
print("\n=== GPT-4o Moderator (per-ticker) ===")
gpt_results = {}
for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    ctx = _build_diag_market_context(ticker)
    try:
        r = openai_mod.review_trade(trade, sample_portfolio, ctx, "diag_test")
        gpt_results[ticker] = r
        print(f"\n  {ticker}:")
        print(f"    verdict   : {r.get('verdict', 'N/A')}")
        print(f"    reasoning : {str(r.get('reasoning', ''))[:120]}")
        risk_flags = r.get("risk_flags", [])
        if risk_flags:
            print(f"    risk_flags: {risk_flags}")
    except Exception as e:
        print(f"\n  {ticker}: ERROR - {e}")
        gpt_results[ticker] = {"error": str(e)}

# --- Test Gemini moderator for each trade ---
print(f"\n=== Gemini Moderator (per-ticker) ===")
gem_results = {}
for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    ctx = _build_diag_market_context(ticker)
    try:
        r = gemini_mod.review_trade(trade, sample_portfolio, ctx, "diag_test")
        gem_results[ticker] = r
        print(f"\n  {ticker}:")
        print(f"    verdict    : {r.get('verdict', 'N/A')}")
        print(f"    assessment : {str(r.get('assessment', ''))[:120]}")
        if r.get("available"):
            print(f"    scores     : growth={r.get('growth_score', 'N/A')} "
                  f"risk={r.get('risk_score', 'N/A')} confidence={r.get('confidence_score', 'N/A')}")
    except Exception as e:
        print(f"\n  {ticker}: ERROR - {e}")
        gem_results[ticker] = {"error": str(e)}

gpt_status = "PASS" if all(r.get("available") for r in gpt_results.values()) else \
    f"WARN ({sum(1 for r in gpt_results.values() if not r.get('available'))} unavailable)"
gem_status = "PASS" if all(r.get("available") for r in gem_results.values()) else \
    f"WARN ({sum(1 for r in gem_results.values() if not r.get('available'))} unavailable)"

print(f"\n=== GPT-4o ({len(gpt_results)} tickers): {gpt_status} ===")
print(f"=== Gemini ({len(gem_results)} tickers): {gem_status} ===")

In [ ]:
from src.agents.moderation.panel import ModerationPanel

print("=== Full Moderation Panel (per-ticker) ===")
print("Testing consensus logic across MULTI_TICKERS...\n")

panel = ModerationPanel()
panel_results = {}

for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    conviction = trade.get("conviction", 60)
    ctx = _build_diag_market_context(ticker)

    try:
        mod_result = panel.review_trade(
            trade_proposal=trade,
            portfolio_context=sample_portfolio,
            market_context=ctx,
            conviction=conviction,
            cycle_id="diag_test",
        )
        panel_results[ticker] = mod_result
        print(f"--- {ticker} (conviction={conviction}) ---")
        print(f"  Consensus            : {mod_result.consensus}")
        print(f"  Strategy verdict     : {mod_result.strategy_verdict}")
        print(f"  Moderators available : {mod_result.moderators_available}")
        print(f"  Caution flag         : {mod_result.caution_flag}")
        if mod_result.gpt4o_verdict:
            print(f"  GPT-4o verdict       : {mod_result.gpt4o_verdict.get('verdict', 'N/A')}")
        if mod_result.gemini_verdict:
            print(f"  Gemini verdict       : {mod_result.gemini_verdict.get('verdict', 'N/A')}")
            print(f"  Gemini scores        : growth={mod_result.gemini_verdict.get('growth_score', 'N/A')} "
                  f"risk={mod_result.gemini_verdict.get('risk_score', 'N/A')} "
                  f"confidence={mod_result.gemini_verdict.get('confidence_score', 'N/A')}")
        print()
    except Exception as e:
        print(f"--- {ticker}: ERROR - {e} ---\n")

# Summary
approved = sum(1 for r in panel_results.values() if r.consensus == "APPROVED")
blocked = sum(1 for r in panel_results.values() if r.consensus == "BLOCKED")
caution = sum(1 for r in panel_results.values() if r.consensus == "CAUTION")
print(f"Summary: {approved} APPROVED, {caution} CAUTION, {blocked} BLOCKED out of {len(panel_results)} tickers")

panel_status = "PASS" if panel_results else "FAIL (no results)"
print(f"\n=== Moderation Panel ({len(panel_results)} tickers): {panel_status} ===")

---
## 14. Risk Manager

In [ ]:
from src.agents.risk.risk_manager import RiskManager

rm = RiskManager()
vix_val = macro.get("vix")

print("=== Individual Risk Rule Checks ===\n")

rules = [
    ("Max single stock", rm.check_max_single_stock("AAPL_US_EQ", 10.0, {})),
    ("Max sector",       rm.check_max_sector("AAPL_US_EQ", "Technology", 10.0, {"Technology": 20.0})),
    ("VIX limit",        rm.check_vix_limit(vix_val, 10.0)),
    ("Cash floor",       rm.check_cash_floor(100.0, 10.0)),
    ("Daily loss halt",  rm.check_daily_loss_halt(0.0, None)),
    ("Drawdown",         rm.check_drawdown(10000, 10500)),
    ("Correlation",      rm.check_correlation({})),
    ("Min positions",    rm.check_min_positions(3, "SELL")),
    ("Cautious state",   rm.check_cautious_state("ACTIVE", "BUY", 10.0)),
]

for name, r in rules:
    icon = "PASS" if r.passed else "FAIL"
    print(f"  [{icon:4s}] {name:20s} : {r.message}")

# --- Multi-ticker trade evaluation (mirrors orchestrator loop) ---
print("\n=== Full Trade Evaluation Across MULTI_TICKERS ===")
print("Simulating sequential evaluations with portfolio state evolving...\n")

# Start with empty portfolio, accumulate positions as trades are approved
running_portfolio = {}
running_sector_allocs = {}
running_cash_pct = 100.0
running_num_positions = 0

risk_verdicts = {}
for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    action = trade.get("action", "BUY")
    target_alloc = trade.get("target_allocation_pct", 8.0)

    # Get sector from stocks_data
    yf_t = ticker.replace("_US_EQ", "").replace("_UK_EQ", "")
    stock = next((s for s in stocks_data if s.get("ticker") in (yf_t, ticker)), {})
    sector = stock.get("fundamentals", {}).get("sector", "Technology")

    verdict = rm.evaluate_trade(
        ticker=ticker,
        action=action,
        proposed_allocation_pct=target_alloc,
        sector=sector,
        current_portfolio=running_portfolio,
        sector_allocations=running_sector_allocs,
        portfolio_returns={},
        current_value=10000,
        peak_value=10000,
        cash_pct=running_cash_pct,
        vix=vix_val,
        daily_pnl_pct=0.0,
        daily_loss_halt_until=None,
        num_positions=running_num_positions,
        system_state="ACTIVE",
        cycle_id="diag_test",
    )
    risk_verdicts[ticker] = verdict

    print(f"--- {ticker} ({action} @ {target_alloc:.1f}%, sector={sector}) ---")
    print(f"  Verdict             : {verdict.verdict}")
    print(f"  Proposed allocation : {verdict.proposed_allocation_pct:.1f}%")
    if verdict.adjusted_allocation_pct:
        print(f"  Adjusted allocation : {verdict.adjusted_allocation_pct:.1f}%")
    print(f"  Rules checked       : {verdict.rules_checked}")
    if verdict.triggered_rules:
        print(f"  Triggered rules     : {verdict.triggered_rules}")
    print(f"  Reasoning           : {verdict.reasoning}")

    # Update running portfolio state if trade was approved
    final_alloc = verdict.adjusted_allocation_pct or target_alloc
    if verdict.verdict in ("APPROVE", "RESIZE"):
        running_portfolio[ticker] = final_alloc
        running_sector_allocs[sector] = running_sector_allocs.get(sector, 0) + final_alloc
        running_cash_pct -= final_alloc
        running_num_positions += 1
    print(f"  Portfolio after     : {running_num_positions} positions, {running_cash_pct:.1f}% cash")
    print()

approved = sum(1 for v in risk_verdicts.values() if v.verdict in ("APPROVE", "RESIZE"))
rejected = sum(1 for v in risk_verdicts.values() if v.verdict == "REJECT")
print(f"Summary: {approved} APPROVE/RESIZE, {rejected} REJECT out of {len(risk_verdicts)} tickers")

print(f"\n=== Risk Manager ({len(risk_verdicts)} tickers): PASS ===")

---
## 15. Trading 212 Client (Practice Mode)

In [ ]:
from src.agents.execution.t212_client import T212Client

print("Testing Trading 212 API (Practice mode)...\n")

try:
    t212 = T212Client()

    print("--- Account Cash ---")
    cash = t212.get_cash()
    print(f"  {json.dumps(cash, indent=2)}")

    print("\n--- Account Info ---")
    info = t212.get_account_info()
    print(f"  {json.dumps(info, indent=2)}")

    print("\n--- Portfolio Positions ---")
    portfolio = t212.get_portfolio()
    print(f"  Positions: {len(portfolio)}")
    for pos in portfolio[:5]:
        print(f"    {pos.get('ticker', 'N/A')}: qty={pos.get('quantity', 0)} @ {pos.get('currentPrice', 0)}")

    # Per-ticker position lookups (mirrors orchestrator per-ticker T212 calls)
    print(f"\n--- Per-Ticker Position Lookups (MULTI_TICKERS) ---")
    for ticker_yf in MULTI_TICKERS:
        ticker_t212 = f"{ticker_yf}_US_EQ"
        try:
            pos = t212.get_position(ticker_t212)
            qty = pos.get("quantity", 0)
            price = pos.get("currentPrice", "N/A")
            print(f"  {ticker_t212}: qty={qty} @ {price}")
        except Exception as e:
            err_str = str(e)
            if "404" in err_str or "not found" in err_str.lower():
                print(f"  {ticker_t212}: no open position (expected if not invested)")
            else:
                print(f"  {ticker_t212}: {e}")

    t212.close()
    t212_status = "PASS"

except Exception as e:
    print(f"  ERROR: {e}")
    print("  (Expected if T212 API keys are not configured or demo server is down)")
    t212_status = f"WARN ({type(e).__name__})"

print(f"\n=== T212 Client: {t212_status} ===")

---
## 16. Order Manager (Dry Run)

In [ ]:
from src.agents.execution.order_manager import OrderManager
from src.agents.execution.t212_client import calculate_quantity

print("=== Order Quantity Calculation ===")
qty = calculate_quantity(target_amount=800.0, price=195.50)
print(f"  Target amount : \u00a3800.00")
print(f"  Price         : $195.50")
print(f"  Calculated qty: {qty}")
assert qty > 0, "Quantity should be positive"

# --- Multi-ticker dry-run order execution (mirrors orchestrator loop) ---
print("\n=== Dry Run Order Execution Across MULTI_TICKERS ===\n")

om = OrderManager(dry_run=True)
exec_results = {}

for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    action = trade.get("action", "BUY")
    target_alloc = trade.get("target_allocation_pct", 8.0)
    conviction = trade.get("conviction", 60)

    # Calculate trade value from allocation
    trade_value = 10000 * target_alloc / 100

    # Get current price from stocks_data
    yf_t = ticker.replace("_US_EQ", "").replace("_UK_EQ", "")
    stock = next((s for s in stocks_data if s.get("ticker") in (yf_t, ticker)), {})
    current_price = float(stock.get("indicators", {}).get("current_price", 195.0))

    # Get moderation/risk results from earlier cells if available
    mod_consensus = panel_results.get(ticker, None)
    mod_str = mod_consensus.consensus if mod_consensus else "APPROVED"
    risk_v = risk_verdicts.get(ticker, None)
    risk_str = risk_v.verdict if risk_v else "APPROVE"

    try:
        exec_result = om.execute_market_order(
            ticker=ticker,
            action=action,
            target_amount_gbp=trade_value,
            current_price=current_price,
            strategy=trade.get("primary_strategy", "unknown"),
            conviction=conviction,
            moderation_result=mod_str,
            risk_result=risk_str,
        )
        exec_results[ticker] = exec_result
        print(f"  {ticker}: status={exec_result['status']:8s} | "
              f"qty={exec_result.get('quantity', 0)} x ${current_price:.2f} = "
              f"\u00a3{exec_result.get('value_gbp', 0):.2f}")
    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")
        exec_results[ticker] = {"status": "error", "error": str(e)}

print(f"\nExecuted {len(exec_results)} orders:")
for ticker, r in exec_results.items():
    assert r["status"] in ("dry_run", "skipped"), f"Unexpected status for {ticker}: {r['status']}"
    print(f"  {ticker}: {r['status']}")

om_status = "PASS" if all(r["status"] == "dry_run" for r in exec_results.values()) else \
    f"WARN ({sum(1 for r in exec_results.values() if r['status'] != 'dry_run')} skipped)"
print(f"\n=== Order Manager ({len(exec_results)} tickers): {om_status} ===")

---
## 17. Trade Journal Generation

In [ ]:
from src.agents.reporting.journal import generate_trade_journal

print("Testing trade journal generation across MULTI_TICKERS...\n")

journal_paths = []

for trade in test_trades:
    ticker = trade.get("ticker", "UNKNOWN")
    action = trade.get("action", "BUY")
    conviction = trade.get("conviction", 60)
    strategy = trade.get("primary_strategy", "unknown")

    # Get per-ticker data from earlier cells
    yf_t = ticker.replace("_US_EQ", "").replace("_UK_EQ", "")
    stock = next((s for s in stocks_data if s.get("ticker") in (yf_t, ticker)), {})
    _ind = stock.get("indicators", {})
    _fund = stock.get("fundamentals", {})
    current_price = float(_ind.get("current_price", 195.0))

    # Get execution result from cell 16
    exec_r = exec_results.get(ticker, {})
    shares = exec_r.get("quantity", 4.0)
    value_gbp = exec_r.get("value_gbp", shares * current_price)

    # Get moderation/risk results from earlier cells
    mod_r = panel_results.get(ticker, None)
    mod_dict = mod_r.to_dict() if mod_r else {}
    risk_v = risk_verdicts.get(ticker, None)
    risk_dict = {"verdict": risk_v.verdict, "rules_checked": risk_v.rules_checked,
                 "triggered_rules": risk_v.triggered_rules, "reasoning": risk_v.reasoning} if risk_v else {}

    try:
        journal_path = generate_trade_journal(
            action=action,
            ticker=ticker,
            shares=shares,
            price=current_price,
            value_gbp=value_gbp,
            weight_pct=trade.get("target_allocation_pct", 8.0),
            conviction=conviction,
            strategy=strategy,
            reasoning=trade.get("reasoning", f"Test trade for {ticker}"),
            growth_potential=trade.get("growth_potential", "MEDIUM"),
            risk_level=trade.get("risk_level", "MEDIUM"),
            catalysts=trade.get("catalysts", ["Earnings growth", "Market momentum"]),
            risks=trade.get("risks", ["Market downturn", "Sector rotation"]),
            exit_conditions=trade.get("exit_conditions", "Stop-loss at -8% or below 50-day MA."),
            upside_target_pct=trade.get("upside_target_pct", 15.0),
            stop_loss_pct=trade.get("stop_loss_pct", -8.0),
            expected_holding_period=trade.get("expected_holding_period", "3-6 months"),
            market_regime=macro.get("market_regime", "SIDEWAYS"),
            vix=macro.get("vix"),
            sp500_trend=str(macro.get("sp500_pct_above_200ma", "N/A")),
            news_sentiment_overall=trade.get("news_sentiment_summary", "Mildly bullish"),
            finnhub_data={},
            alpha_vantage_data=globals().get("av_broad_sentiment", {}),
            moderation_results=mod_dict,
            risk_verdict=risk_dict,
            indicators=_ind,
            fundamentals=_fund,
            portfolio_state={"total_value": 10000, "cash": 10000 - value_gbp,
                             "invested": value_gbp, "num_positions": 1,
                             "total_return_pct": 0, "alpha_pct": 0, "positions": []},
        )

        assert os.path.exists(journal_path), f"Journal file not created for {ticker}!"
        size = os.path.getsize(journal_path)
        journal_paths.append(journal_path)
        print(f"  {ticker}: {journal_path} ({size:,} bytes)")

    except Exception as e:
        print(f"  {ticker}: ERROR - {e}")
        import traceback; traceback.print_exc()

journal_status = "PASS" if len(journal_paths) == len(test_trades) else \
    f"WARN ({len(journal_paths)}/{len(test_trades)} journals created)"
print(f"\n=== Trade Journal ({len(journal_paths)} journals): {journal_status} ===")

---
## 18. Full Orchestrator Dry Run

**Cost:** ~$0.03-0.05 (Anthropic + OpenAI + Google combined).

In [ ]:
from src.orchestrator.main import Orchestrator

print("Running FULL orchestrator cycle (dry_run=True)...")
print("This tests the complete pipeline end-to-end.\n")

try:
    orch = Orchestrator(dry_run=True)
    cycle_result = orch.run_cycle()

    print(f"Cycle ID     : {cycle_result.get('cycle_id', 'N/A')}")
    print(f"Status       : {cycle_result.get('status', 'N/A')}")
    print(f"Trades       : {cycle_result.get('num_trades', 0)}")
    print(f"Errors       : {cycle_result.get('errors', [])}")

    if cycle_result.get("trades"):
        print("\n--- Trade Details ---")
        for t in cycle_result["trades"]:
            print(f"  {t['ticker']}: {t['action']} @ {t['allocation_pct']:.1f}% | mod={t['moderation']} risk={t['risk']}")

    cost = cycle_result.get("cost_summary", {})
    if cost:
        print(f"\n--- Cost Summary (today) ---")
        for k, v in cost.items():
            print(f"  {k:15s} : \u00a3{v:.4f}")

    orch.close()
    orch_status = "PASS" if cycle_result.get("status") == "completed" else f"WARN ({cycle_result.get('status')})"

except Exception as e:
    print(f"  ERROR: {e}")
    import traceback; traceback.print_exc()
    orch_status = f"FAIL ({e})"

print(f"\n=== Orchestrator Dry Run: {orch_status} ===")

---
## 19. Performance Tracker & Trade Outcomes

**Expected:** Metrics computed from `portfolio_snapshots` and `trade_outcomes`; CLI-style summary. If no snapshots/orders exist, metrics may be empty or zeros — still **PASS** if no exception.

In [ ]:
from src.agents.reporting.performance_tracker import update_performance_metrics
from src.agents.reporting.trade_outcome_tracker import update_trade_outcomes
from src.data.database import get_session
from src.data.models import PerformanceMetric, TradeOutcome

session = get_session()
try:
    n_outcomes = update_trade_outcomes(session=session)
    n_perf = update_performance_metrics(session=session)
    print(f"Trade outcomes created this run: {n_outcomes}")
    print(f"Performance metrics rows written: {n_perf}")

    latest = session.query(PerformanceMetric).order_by(PerformanceMetric.snapshot_date.desc()).first()
    if latest:
        print("\n=== Latest performance_metrics ===")
        print(f"  snapshot_date     : {latest.snapshot_date}")
        print(f"  sharpe_30d/60d/90d: {latest.sharpe_30d}, {latest.sharpe_60d}, {latest.sharpe_90d}")
        print(f"  max_drawdown_pct  : {latest.max_drawdown_pct}")
        print(f"  win_rate (mom/mr/factor): {latest.win_rate_momentum}, {latest.win_rate_mean_reversion}, {latest.win_rate_factor}")
        print(f"  num_trades        : {latest.num_trades}")
    else:
        print("\nNo performance_metrics rows yet (need portfolio_snapshots + optional trade_outcomes).")

    n_to = session.query(TradeOutcome).count()
    print(f"\nTotal trade_outcomes rows: {n_to}")
finally:
    session.close()

print("\n=== Performance & Trade Outcomes: PASS ===")

---
## 20. Notifications (Chat / Alerts)

**Expected:** NotificationService loads; routes and channels from config. No real send unless you have webhook/SMTP configured. **PASS** if service and formatters run without error.

In [ ]:
from src.agents.notifications import NotificationService
from src.agents.notifications.types import NotificationEvent
from src.agents.notifications.formatters import render_event
from datetime import datetime, timezone

svc = NotificationService()
print("=== Notification service ===")
print(f"  enabled : {svc.enabled}")
print(f"  routes  : {list(svc.routes.keys())}")
print(f"  channels: {svc.default_channels}")

# Build a minimal event and render (no send)
event = NotificationEvent(
    event_id="diag-test",
    event_type="trade_instruction_approved",
    occurred_at=datetime.now(timezone.utc),
    cycle_id="diag_cycle",
    severity="info",
    source="diagnostics",
    dedup_key="diag-key",
    payload={"ticker": "AAPL_US_EQ", "action": "BUY", "cycle_id": "diag_cycle", "dry_run": True},
)
for ch in ["slack", "email"]:
    msgs = render_event(event, ch)
    print(f"  {ch}: {len(msgs)} message(s), first subject/body len: {msgs[0].subject[:50] if msgs else 'N/A'} / {len(msgs[0].body) if msgs else 0}")

print("\n=== Notifications: PASS ===")

---
## 21. Backtesting

**Expected:** Synthetic bars generated; backtest engine runs; metrics and equity curve returned. **PASS** if no error and `metrics` (e.g. sharpe, max_drawdown_pct) or equity_curve length >= 2.

In [ ]:
import pandas as pd
from src.backtesting.engine import BacktestEngine
from src.backtesting.io import generate_synthetic_bars

bars = {
    "AAPL": generate_synthetic_bars("AAPL", pd.Timestamp("2023-01-01"), pd.Timestamp("2023-06-30"), seed=42),
    "MSFT": generate_synthetic_bars("MSFT", pd.Timestamp("2023-01-01"), pd.Timestamp("2023-06-30"), seed=43),
}
config = {"seed": 42, "initial_cash": 10000.0, "slippage_bps": 5.0, "max_positions": 5, "sma_period": 20}
engine = BacktestEngine(config, seed=42)
result = engine.run(bars, benchmark=None)

assert "error" not in result, result.get("error", "Unknown error")
print("=== Backtest result ===")
print(f"  equity_curve points : {len(result['equity_curve'])}")
print(f"  trades              : {len(result['trades'])}")
print(f"  metrics             : {result.get('metrics', {})}")
assert len(result["equity_curve"]) >= 2
print("\n=== Backtesting: PASS ===")

---
## 22. Walk-Forward Validation

**Expected:** Splits created; walk-forward run on synthetic bars; aggregate metrics and recommendation (safe to deploy / hold). **PASS** if no exception and recommendation is one of the two.

In [ ]:
from src.backtesting.walk_forward import make_splits, run_walk_forward, aggregate_fold_metrics
from src.backtesting.promotion_report import write_promotion_report
import tempfile
from pathlib import Path

bars_wf = {
    "AAPL": generate_synthetic_bars("AAPL", pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31"), seed=1),
    "SPY": generate_synthetic_bars("SPY", pd.Timestamp("2022-01-01"), pd.Timestamp("2023-12-31"), seed=2),
}
config_wf = {"seed": 1, "initial_cash": 10000.0}
splits = make_splits("2022-01-01", "2023-12-31", n_folds=2, test_days=252)
fold_results = run_walk_forward(config_wf, ["AAPL", "SPY"], splits, bars_cache=bars_wf)
assert len(fold_results) >= 1, "Walk-forward should produce at least one fold"

agg = aggregate_fold_metrics(fold_results)
print("=== Walk-forward aggregate ===")
for k, v in list(agg.items())[:12]:
    print(f"  {k}: {v}")

with tempfile.TemporaryDirectory() as tmp:
    rec = write_promotion_report(agg, fold_results, Path(tmp) / "promotion_report.md")
print(f"\nRecommendation: {rec}")
assert rec in ("safe to deploy", "hold")
print("\n=== Walk-Forward Validation: PASS ===")

---
## 23. Post-Run Database Inspection

In [ ]:
from sqlalchemy import text
from src.data.database import get_session

session = get_session()
print("=== Recent Database Activity ===\n")

queries = {
    "Strategy decisions (last 5)": "SELECT timestamp, cycle_id, ticker, action, conviction, target_allocation_pct FROM strategy_decisions ORDER BY timestamp DESC LIMIT 5",
    "Moderation logs (last 5)": "SELECT timestamp, cycle_id, ticker, moderator, verdict, consensus FROM moderation_logs ORDER BY timestamp DESC LIMIT 5",
    "Risk decisions (last 5)": "SELECT timestamp, cycle_id, ticker, proposed_action, verdict, reasoning FROM risk_decisions ORDER BY timestamp DESC LIMIT 5",
    "Orders (last 5)": "SELECT timestamp, ticker, action, quantity, price, status, strategy FROM orders ORDER BY timestamp DESC LIMIT 5",
    "Cost log (last 10)": "SELECT timestamp, provider, model, input_tokens, output_tokens, cost_gbp, purpose FROM cost_logs ORDER BY timestamp DESC LIMIT 10",
    "API logs (last 5)": "SELECT timestamp, service, method, endpoint, status_code, duration_ms FROM api_logs ORDER BY timestamp DESC LIMIT 5",
}

try:
    for title, sql in queries.items():
        print(f"--- {title} ---")
        try:
            rows = session.execute(text(sql)).fetchall()
            if rows:
                for row in rows:
                    print(f"  {row}")
            else:
                print("  (no rows)")
        except Exception as e:
            print(f"  Query error: {e}")
        print()
finally:
    session.close()

print("=== Database Inspection: DONE ===")

---
## 24. Summary Report

In [ ]:
print("=" * 60)
print("  DIAGNOSTICS SUMMARY")
print("=" * 60)
print(f"  Timestamp : {datetime.utcnow().isoformat()}Z")
print("-" * 60)

results = {
    "1.  Configuration"      : "PASS",
    "2.  Database"           : "PASS",
    "3.  State Machine"      : "PASS",
    "4.  Cost Tracker"       : "PASS",
    "5.  yfinance OHLCV"    : "PASS",
    "6.  Indicators"         : "PASS",
    "7.  Fundamentals"       : "PASS",
    "7.5 Macro Intelligence" : globals().get('macro_intel_status', 'NOT RUN'),
    "8.  Macro Data"         : "PASS",
    "9.  Finnhub API"        : globals().get('finnhub_status', 'NOT RUN'),
    "10. Alpha Vantage API"  : globals().get('av_status', 'NOT RUN'),
    "11. Sub-Strategies"     : "PASS",
    "12. Claude Synthesis"   : globals().get('claude_status', 'NOT RUN'),
    "13a. GPT-4o Moderator" : globals().get('gpt_status', 'NOT RUN'),
    "13b. Gemini Moderator" : globals().get('gem_status', 'NOT RUN'),
    "13c. Moderation Panel" : globals().get('panel_status', 'NOT RUN'),
    "14. Risk Manager"       : "PASS",
    "15. T212 Client"        : globals().get('t212_status', 'NOT RUN'),
    "16. Order Manager"      : globals().get('om_status', 'NOT RUN'),
    "17. Trade Journal"      : globals().get('journal_status', 'NOT RUN'),
    "18. Orchestrator"       : globals().get('orch_status', 'NOT RUN'),
}

pass_count = sum(1 for v in results.values() if v == "PASS")
warn_count = sum(1 for v in results.values() if v.startswith("WARN"))
fail_count = sum(1 for v in results.values() if v.startswith("FAIL"))
not_run    = sum(1 for v in results.values() if v == "NOT RUN")

for name, result in results.items():
    if result == "PASS":
        icon = "[OK]"
    elif result.startswith("WARN"):
        icon = "[!!]"
    elif result.startswith("FAIL"):
        icon = "[XX]"
    else:
        icon = "[--]"
    print(f"  {icon} {name:28s} {result}")

print("-" * 60)
print(f"  PASS: {pass_count}  |  WARN: {warn_count}  |  FAIL: {fail_count}  |  NOT RUN: {not_run}")
print("=" * 60)

if fail_count == 0:
    print("\n  All components operational. Ready for live run.")
else:
    print(f"\n  {fail_count} component(s) failed. Review errors above.")